# Assignment 1: Model Comparison and Analysis
## FreshRetailNet-50K Demand Forecasting

**Objective:** Compare baseline forecasting models using comprehensive evaluation metrics and error analysis to identify the best-performing approach.

### Workflow:
1. Train all baseline models (naive, statistical, ML-based)
2. Evaluate models on test set with all metrics
3. Create comparison tables and visualizations
4. Perform detailed error analysis
5. Generate actionable insights

## 1. Setup and Data Preparation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import yaml
import warnings
import sys
from datetime import datetime
import joblib

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Add src to path
sys.path.append('src')

from data.data_loader import FreshRetailDataLoader
from data.feature_engineering import FeatureEngineer
from models.baseline.linear_models import LinearForecastingModel
from models.baseline.tree_models import TreeForecastingModel
from models.baseline.naive_models import SimpleNaiveModel, SeasonalNaiveModel
from evaluate.metrics import ForecastingMetrics

# Load configuration
with open('config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("✓ All imports successful")
print("✓ Configuration loaded")

✓ All imports successful
✓ Configuration loaded


In [2]:
# Load and prepare data
loader = FreshRetailDataLoader()
train_data, eval_data = loader.load_data()

print(f"\n📊 Data Loaded:")
print(f"  Training: {train_data.shape[0]:,} samples × {train_data.shape[1]} features")
print(f"  Evaluation: {eval_data.shape[0]:,} samples × {eval_data.shape[1]} features")

# Feature engineering with consistent aggregations
print("\n⚙️ Feature Engineering...")
feature_engineer = FeatureEngineer(config['preprocessing'])

# Compute aggregations from training data
target_col = config['data']['target_column']
store_stats = train_data.groupby('store_id').agg({
    'sale_amount': ['mean', 'std', 'max'],
    'product_id': 'nunique'
}).round(2)
store_stats.columns = ['store_avg_sales', 'store_sales_volatility', 'store_max_sales', 'store_product_count']

product_stats = train_data.groupby('product_id').agg({
    'sale_amount': ['mean', 'std'],
    'store_id': 'nunique'
}).round(2)
product_stats.columns = ['product_avg_sales', 'product_sales_volatility', 'product_store_count']

city_stats = train_data.groupby('city_id').agg({
    'sale_amount': 'mean',
    'hours_stock_status': 'mean'
}).round(2)
city_stats.columns = ['city_avg_sales', 'city_stock_availability']

# Apply aggregations
def apply_aggregations(df):
    df = df.copy()
    df = df.merge(store_stats, on='store_id', how='left')
    df = df.merge(product_stats, on='product_id', how='left')
    df = df.merge(city_stats, on='city_id', how='left')
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].mean())
    return df

# Apply feature engineering
train_engineered = feature_engineer.engineer_all_features(train_data, skip_categorical=True)
train_engineered = apply_aggregations(train_engineered)

eval_engineered = feature_engineer.engineer_all_features(eval_data, skip_categorical=True)
eval_engineered = apply_aggregations(eval_engineered)

print(f"✓ Feature engineering complete: {train_engineered.shape[1]} features")

# Prepare features and targets
metadata_cols = ['store_id', 'product_id', 'city_id', 'dt', target_col]
feature_cols = [col for col in train_engineered.columns if col not in metadata_cols]

X_train = train_engineered[feature_cols].fillna(0)
y_train = train_engineered[target_col]
X_eval = eval_engineered[feature_cols].fillna(0)
y_eval = eval_engineered[target_col]

print(f"✓ Training features shape: {X_train.shape}")
print(f"✓ Evaluation features shape: {X_eval.shape}")


📊 Data Loaded:
  Training: 450,000 samples × 19 features
  Evaluation: 35,000 samples × 19 features

⚙️ Feature Engineering...
✓ Feature engineering complete: 75 features
✓ Feature engineering complete: 75 features
✓ Training features shape: (450000, 70)
✓ Evaluation features shape: (35000, 70)
✓ Training features shape: (450000, 70)
✓ Evaluation features shape: (35000, 70)


## 2. Train Baseline Models

In [3]:
print("=" * 80)
print("TRAINING BASELINE MODELS")
print("=" * 80)

# Define models to train
models_config = {
    'linear': LinearForecastingModel(config=config['models']['baseline'].get('linear', {})),
    'tree': TreeForecastingModel(config=config['models']['baseline'].get('tree', {})),
    'naive': SimpleNaiveModel(config=config['models']['baseline'].get('naive', {})),
    'seasonal_naive': SeasonalNaiveModel(config=config['models']['baseline'].get('seasonal_naive', {}))
}

# Store trained models
trained_models = {}
training_times = {}

print(f"\nTraining {len(models_config)} models...\n")

for model_name, model in models_config.items():
    try:
        print(f"📌 Training {model_name}...", end=" ")
        start_time = datetime.now()
        model.fit(X_train, y_train)
        training_time = (datetime.now() - start_time).total_seconds()
        training_times[model_name] = training_time
        trained_models[model_name] = model
        print(f"✓ Done ({training_time:.2f}s)")
    except Exception as e:
        print(f"✗ Failed: {str(e)[:50]}")

print(f"\n✓ Successfully trained {len(trained_models)}/{len(models_config)} models")

TRAINING BASELINE MODELS

Training 4 models...

📌 Training linear... ✓ Done (2.48s)
📌 Training tree... ✓ Done (2.48s)
📌 Training tree... ✓ Done (18.82s)
📌 Training naive... ✓ Done (18.82s)
📌 Training naive... ✗ Failed: X must contain columns: ['store_id', 'product_id']
📌 Training seasonal_naive... ✗ Failed: X must contain 'dt' column for temporal informatio

✓ Successfully trained 2/4 models
✗ Failed: X must contain columns: ['store_id', 'product_id']
📌 Training seasonal_naive... ✗ Failed: X must contain 'dt' column for temporal informatio

✓ Successfully trained 2/4 models


## 3. Evaluate Models on Test Set

In [4]:
print("=" * 80)
print("EVALUATING MODELS ON TEST SET")
print("=" * 80)

# Generate predictions
results = {}
predictions = {}
inference_times = {}

for model_name, model in trained_models.items():
    try:
        print(f"\n📊 Evaluating {model_name}...", end=" ")
        
        # Make predictions
        start_time = datetime.now()
        y_pred_train = model.predict(X_train)
        y_pred_eval = model.predict(X_eval)
        inference_time = (datetime.now() - start_time).total_seconds()
        inference_times[model_name] = inference_time
        
        # Calculate metrics
        train_metrics = ForecastingMetrics.calculate_all_metrics(y_train.values, y_pred_train)
        eval_metrics = ForecastingMetrics.calculate_all_metrics(y_eval.values, y_pred_eval, y_train.values)
        
        # Store results
        results[model_name] = {
            'train_metrics': train_metrics,
            'eval_metrics': eval_metrics,
            'training_time': training_times[model_name],
            'inference_time': inference_time
        }
        
        predictions[model_name] = {
            'y_train_pred': y_pred_train,
            'y_eval_pred': y_pred_eval,
            'residuals_train': y_train.values - y_pred_train,
            'residuals_eval': y_eval.values - y_pred_eval
        }
        
        print(f"✓")
        print(f"  - Train RMSE: {train_metrics['RMSE']:.4f}")
        print(f"  - Eval RMSE:  {eval_metrics['RMSE']:.4f}")
        print(f"  - Eval MAE:   {eval_metrics['MAE']:.4f}")
        print(f"  - Eval MAPE:  {eval_metrics['MAPE']:.2f}%")
        
    except Exception as e:
        print(f"✗ Failed: {str(e)[:60]}")

EVALUATING MODELS ON TEST SET

📊 Evaluating linear... 

✓
  - Train RMSE: 0.5253
  - Eval RMSE:  0.4712
  - Eval MAE:   0.3344
  - Eval MAPE:  32513295.39%

📊 Evaluating tree... ✓
  - Train RMSE: 0.3251
  - Eval RMSE:  0.4090
  - Eval MAE:   0.0888
  - Eval MAPE:  29695827.14%
✓
  - Train RMSE: 0.3251
  - Eval RMSE:  0.4090
  - Eval MAE:   0.0888
  - Eval MAPE:  29695827.14%


## 4. Model Comparison Table

In [5]:
print("=" * 80)
print("MODEL COMPARISON - COMPREHENSIVE TABLE")
print("=" * 80)

# Create comprehensive comparison dataframe
comparison_data = []

for model_name in results.keys():
    eval_metrics = results[model_name]['eval_metrics']
    train_metrics = results[model_name]['train_metrics']
    
    comparison_data.append({
        'Model': model_name.upper(),
        'Train_R²': train_metrics.get('R²', np.nan),
        'Eval_R²': eval_metrics.get('R²', np.nan),
        'Train_RMSE': train_metrics.get('RMSE', np.nan),
        'Eval_RMSE': eval_metrics.get('RMSE', np.nan),
        'Train_MAE': train_metrics.get('MAE', np.nan),
        'Eval_MAE': eval_metrics.get('MAE', np.nan),
        'Eval_MAPE': eval_metrics.get('MAPE', np.nan),
        'Eval_sMAPE': eval_metrics.get('sMAPE', np.nan),
        'Eval_Bias': eval_metrics.get('Bias', np.nan),
        'Train_Time_sec': results[model_name]['training_time'],
        'Inference_Time_sec': results[model_name]['inference_time']
    })

comparison_df = pd.DataFrame(comparison_data).set_index('Model')

print("\n🏆 Overall Performance Metrics:")
print(comparison_df.round(4))

# Ranking by different metrics
print("\n" + "=" * 80)
print("RANKINGS BY METRIC")
print("=" * 80)

ranking_metrics = ['Eval_RMSE', 'Eval_MAE', 'Eval_MAPE', 'Eval_R²']
for metric in ranking_metrics:
    if metric in comparison_df.columns:
        if 'RMSE' in metric or 'MAE' in metric or 'MAPE' in metric:
            ranking = comparison_df[metric].rank()
        else:
            ranking = comparison_df[metric].rank(ascending=False)
        
        print(f"\n📊 {metric}:")
        sorted_ranking = ranking.sort_values()
        for rank, (model, value) in enumerate(sorted_ranking.items(), 1):
            metric_value = comparison_df.loc[model, metric]
            print(f"   {rank}. {model}: {metric_value:.4f}")

# Best model by each metric
print("\n" + "=" * 80)
print("BEST MODEL BY METRIC")
print("=" * 80)

for metric in ranking_metrics:
    if metric in comparison_df.columns:
        if 'RMSE' in metric or 'MAE' in metric or 'MAPE' in metric:
            best_model = comparison_df[metric].idxmin()
        else:
            best_model = comparison_df[metric].idxmax()
        best_value = comparison_df.loc[best_model, metric]
        print(f"✓ {metric}: {best_model} ({best_value:.4f})")

MODEL COMPARISON - COMPREHENSIVE TABLE

🏆 Overall Performance Metrics:
        Train_R²  Eval_R²  Train_RMSE  Eval_RMSE  Train_MAE  Eval_MAE  \
Model                                                                   
LINEAR       NaN      NaN      0.5253     0.4712     0.2985    0.3344   
TREE         NaN      NaN      0.3251     0.4090     0.2010    0.0888   

           Eval_MAPE  Eval_sMAPE  Eval_Bias  Train_Time_sec  \
Model                                                         
LINEAR  3.251330e+07     80.4550    -0.2282          2.4838   
TREE    2.969583e+07     15.8386    -0.0288         18.8201   

        Inference_Time_sec  
Model                       
LINEAR              1.2982  
TREE                1.5812  

RANKINGS BY METRIC

📊 Eval_RMSE:
   1. TREE: 0.4090
   2. LINEAR: 0.4712

📊 Eval_MAE:
   1. TREE: 0.0888
   2. LINEAR: 0.3344

📊 Eval_MAPE:
   1. TREE: 29695827.1404
   2. LINEAR: 32513295.3895

📊 Eval_R²:
   1. LINEAR: nan
   2. TREE: nan

BEST MODEL BY METRIC
✓ Ev

ValueError: Encountered all NA values

In [ ]:
# Visualize model comparisons
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# RMSE Comparison
models = comparison_df.index.tolist()
x = np.arange(len(models))
width = 0.35

axes[0, 0].bar(x - width/2, comparison_df['Train_RMSE'], width, label='Train RMSE', alpha=0.8, color='steelblue')
axes[0, 0].bar(x + width/2, comparison_df['Eval_RMSE'], width, label='Eval RMSE', alpha=0.8, color='coral')
axes[0, 0].set_ylabel('RMSE (Lower is Better)', fontweight='bold')
axes[0, 0].set_title('Model Comparison: RMSE', fontsize=12, fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(models, rotation=45)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='y')

# MAE Comparison
axes[0, 1].bar(x - width/2, comparison_df['Train_MAE'], width, label='Train MAE', alpha=0.8, color='lightgreen')
axes[0, 1].bar(x + width/2, comparison_df['Eval_MAE'], width, label='Eval MAE', alpha=0.8, color='salmon')
axes[0, 1].set_ylabel('MAE (Lower is Better)', fontweight='bold')
axes[0, 1].set_title('Model Comparison: MAE', fontsize=12, fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(models, rotation=45)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3, axis='y')

# R² Comparison
axes[1, 0].bar(x, comparison_df['Eval_R²'], alpha=0.8, color='purple', edgecolor='black')
axes[1, 0].set_ylabel('R² Score (Higher is Better)', fontweight='bold')
axes[1, 0].set_title('Model Comparison: R² Score', fontsize=12, fontweight='bold')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(models, rotation=45)
axes[1, 0].axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Training Time Comparison
axes[1, 1].bar(x, comparison_df['Train_Time_sec'], alpha=0.8, color='darkblue', edgecolor='black')
axes[1, 1].set_ylabel('Training Time (seconds)', fontweight='bold')
axes[1, 1].set_title('Model Comparison: Training Time', fontsize=12, fontweight='bold')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(models, rotation=45)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Error Analysis - Performance by Segments

In [ ]:
print("=" * 80)
print("ERROR ANALYSIS BY SEGMENTS")
print("=" * 80)

# Select best model for analysis
best_eval_rmse_model = comparison_df['Eval_RMSE'].idxmin()
best_model_name = best_eval_rmse_model.lower()
best_model = trained_models[best_model_name]

print(f"\nAnalyzing best model: {best_eval_rmse_model}")
print(f"  - Eval RMSE: {results[best_model_name]['eval_metrics']['RMSE']:.4f}")
print(f"  - Eval MAE: {results[best_model_name]['eval_metrics']['MAE']:.4f}")

# Add predictions to evaluation data for analysis
eval_analysis = eval_engineered.copy()
eval_analysis['y_true'] = y_eval.values
eval_analysis['y_pred'] = predictions[best_model_name]['y_eval_pred']
eval_analysis['residuals'] = predictions[best_model_name]['residuals_eval']
eval_analysis['abs_error'] = np.abs(eval_analysis['residuals'])
eval_analysis['mape'] = np.abs((eval_analysis['y_true'] - eval_analysis['y_pred']) / (eval_analysis['y_true'] + 1e-8)) * 100

# Convert datetime for analysis
eval_analysis['dt'] = pd.to_datetime(eval_analysis['dt'])
eval_analysis['hour'] = eval_analysis['dt'].dt.hour
eval_analysis['day_of_week'] = eval_analysis['dt'].dt.dayofweek
eval_analysis['month'] = eval_analysis['dt'].dt.month

# Analysis by store
print("\n🏪 Performance by Store (Top 10 by Error):")
store_errors = eval_analysis.groupby('store_id').agg({
    'y_true': 'count',
    'abs_error': ['mean', 'std'],
    'mape': 'mean'
}).round(4)
store_errors.columns = ['Count', 'MAE', 'Std', 'MAPE']
worst_stores = store_errors.nlargest(10, 'MAE')
print(worst_stores)

# Analysis by hour
print("\n⏰ Performance by Hour:")
hour_errors = eval_analysis.groupby('hour').agg({
    'abs_error': 'mean',
    'y_true': 'count'
}).round(4)
hour_errors.columns = ['MAE', 'Count']
print(hour_errors)

# Analysis by day of week
print("\n📅 Performance by Day of Week:")
dow_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_errors = eval_analysis.groupby('day_of_week').agg({
    'abs_error': 'mean',
    'y_true': 'count'
}).round(4)
dow_errors.columns = ['MAE', 'Count']
dow_errors.index = dow_names
print(dow_errors)

# Analysis by month
print("\n📊 Performance by Month:")
month_errors = eval_analysis.groupby('month').agg({
    'abs_error': 'mean',
    'y_true': 'count'
}).round(4)
month_errors.columns = ['MAE', 'Count']
print(month_errors)

# Visualize error analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Error by hour
axes[0, 0].plot(hour_errors.index, hour_errors['MAE'], marker='o', linewidth=2, color='steelblue', markersize=8)
axes[0, 0].fill_between(hour_errors.index, hour_errors['MAE'], alpha=0.3, color='steelblue')
axes[0, 0].set_title(f'{best_eval_rmse_model} - Mean Error by Hour', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('MAE')
axes[0, 0].grid(True, alpha=0.3)

# Error by day of week
axes[0, 1].bar(range(len(dow_errors)), dow_errors['MAE'].values, color='coral', alpha=0.7, edgecolor='black')
axes[0, 1].set_xticks(range(len(dow_errors)))
axes[0, 1].set_xticklabels(dow_errors.index, rotation=45)
axes[0, 1].set_title(f'{best_eval_rmse_model} - Mean Error by Day of Week', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('MAE')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Error by month
axes[1, 0].plot(month_errors.index, month_errors['MAE'], marker='s', linewidth=2, color='darkgreen', markersize=8)
axes[1, 0].fill_between(month_errors.index, month_errors['MAE'], alpha=0.3, color='lightgreen')
axes[1, 0].set_title(f'{best_eval_rmse_model} - Mean Error by Month', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('MAE')
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].grid(True, alpha=0.3)

# Error distribution
axes[1, 1].hist(eval_analysis['abs_error'], bins=50, color='purple', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(x=eval_analysis['abs_error'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {eval_analysis["abs_error"].mean():.4f}')
axes[1, 1].axvline(x=eval_analysis['abs_error'].median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {eval_analysis["abs_error"].median():.4f}')
axes[1, 1].set_title(f'{best_eval_rmse_model} - Distribution of Absolute Errors', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Absolute Error')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Residual Analysis

In [ ]:
print("=" * 80)
print("RESIDUAL ANALYSIS")
print("=" * 80)

residuals = predictions[best_model_name]['residuals_eval']

print(f"\n📊 Residual Statistics:")
print(f"  • Mean (Bias): {residuals.mean():.6f}")
print(f"  • Std Dev: {residuals.std():.6f}")
print(f"  • Min: {residuals.min():.6f}")
print(f"  • Max: {residuals.max():.6f}")
print(f"  • Skewness: {residuals.skew():.6f}")
print(f"  • Kurtosis: {residuals.kurtosis():.6f}")

# Create comprehensive residual plots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Residuals vs Fitted Values
axes[0, 0].scatter(eval_analysis['y_pred'], residuals, alpha=0.5, s=10, color='steelblue')
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_title('Residuals vs Fitted Values', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].grid(True, alpha=0.3)

# Q-Q Plot (Normal probability plot)
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot (Normality Check)', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Histogram of Residuals
axes[0, 2].hist(residuals, bins=50, color='coral', alpha=0.7, edgecolor='black')
axes[0, 2].axvline(x=residuals.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {residuals.mean():.4f}')
axes[0, 2].set_title('Distribution of Residuals', fontsize=12, fontweight='bold')
axes[0, 2].set_xlabel('Residuals')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3, axis='y')

# Scale-Location Plot (sqrt of standardized residuals)
standardized_residuals = residuals / residuals.std()
axes[1, 0].scatter(eval_analysis['y_pred'], np.sqrt(np.abs(standardized_residuals)), alpha=0.5, s=10, color='darkgreen')
axes[1, 0].set_title('Scale-Location Plot', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Fitted Values')
axes[1, 0].set_ylabel('√|Standardized Residuals|')
axes[1, 0].grid(True, alpha=0.3)

# Residuals over Time
sample_indices = np.arange(0, len(residuals), max(1, len(residuals)//1000))
axes[1, 1].plot(sample_indices, residuals.iloc[sample_indices], linewidth=0.5, color='purple', alpha=0.7)
axes[1, 1].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[1, 1].set_title('Residuals Over Time (Sample)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Sample Index')
axes[1, 1].set_ylabel('Residuals')
axes[1, 1].grid(True, alpha=0.3)

# Box plot of Residuals
axes[1, 2].boxplot(residuals, vert=True)
axes[1, 2].set_title('Box Plot of Residuals', fontsize=12, fontweight='bold')
axes[1, 2].set_ylabel('Residuals')
axes[1, 2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Normality test
from scipy.stats import shapiro, normaltest
try:
    stat_shapiro, p_shapiro = shapiro(residuals.sample(min(5000, len(residuals))))
    print(f"\n📈 Normality Tests:")
    print(f"  • Shapiro-Wilk Test: statistic={stat_shapiro:.6f}, p-value={p_shapiro:.6f}")
    print(f"    → Residuals are {'normally distributed' if p_shapiro > 0.05 else 'NOT normally distributed'} (α=0.05)")
except:
    print("  • Shapiro-Wilk test skipped (sample too large)")

## 7. Predictions Visualization

In [ ]:
# Plot predictions vs actual values
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Scatter plot: Predicted vs Actual
axes[0, 0].scatter(eval_analysis['y_true'], eval_analysis['y_pred'], alpha=0.3, s=10, color='steelblue')
min_val = min(eval_analysis['y_true'].min(), eval_analysis['y_pred'].min())
max_val = max(eval_analysis['y_true'].max(), eval_analysis['y_pred'].max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0, 0].set_title(f'{best_eval_rmse_model} - Predicted vs Actual Values', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Actual Values')
axes[0, 0].set_ylabel('Predicted Values')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Time series sample
sample_size = min(500, len(eval_analysis))
sample_indices = np.random.choice(len(eval_analysis), sample_size, replace=False)
sample_indices = np.sort(sample_indices)
sample_eval = eval_analysis.iloc[sample_indices].copy()

axes[0, 1].plot(range(len(sample_eval)), sample_eval['y_true'].values, label='Actual', linewidth=1.5, color='blue', alpha=0.8)
axes[0, 1].plot(range(len(sample_eval)), sample_eval['y_pred'].values, label='Predicted', linewidth=1, color='red', alpha=0.8, linestyle='--')
axes[0, 1].fill_between(range(len(sample_eval)), sample_eval['y_true'].values, sample_eval['y_pred'].values, alpha=0.2, color='gray')
axes[0, 1].set_title(f'{best_eval_rmse_model} - Sample Time Series (500 points)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Sample Index')
axes[0, 1].set_ylabel('Sales Amount')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Error percentage distribution
axes[1, 0].hist(eval_analysis['mape'], bins=50, color='coral', alpha=0.7, edgecolor='black')
axes[1, 0].axvline(x=eval_analysis['mape'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {eval_analysis["mape"].mean():.2f}%')
axes[1, 0].set_title(f'{best_eval_rmse_model} - MAPE Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('MAPE (%)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')
axes[1, 0].set_xlim(left=0)

# Prediction error quantiles
quantiles = np.linspace(0, 1, 11)
error_quantiles = np.quantile(np.abs(residuals), quantiles)
axes[1, 1].plot(quantiles * 100, error_quantiles, marker='o', linewidth=2, markersize=8, color='darkgreen')
axes[1, 1].fill_between(quantiles * 100, error_quantiles, alpha=0.3, color='lightgreen')
axes[1, 1].set_title(f'{best_eval_rmse_model} - Absolute Error Quantiles', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Quantile (%)')
axes[1, 1].set_ylabel('Absolute Error')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Key Findings and Recommendations

### 🏆 Best Model: **{{ best_model_name.upper() }}**

**Performance Metrics:**
- **Evaluation RMSE:** {{ results[best_model_name]['eval_metrics']['RMSE']:.4f }} (Root Mean Squared Error)
- **Evaluation MAE:** {{ results[best_model_name]['eval_metrics']['MAE']:.4f }} (Mean Absolute Error)
- **Evaluation MAPE:** {{ results[best_model_name]['eval_metrics']['MAPE']:.2f }}% (Mean Absolute Percentage Error)
- **R² Score:** {{ results[best_model_name]['eval_metrics']['R²']:.4f }}
- **Training Time:** {{ results[best_model_name]['training_time']:.2f }} seconds

### 📊 Model Strengths and Weaknesses

#### Strengths:
1. **Captures Complex Patterns:** ML-based models learn non-linear relationships better than naive approaches
2. **Handles Multiple Features:** Utilizes engineered features (temporal, store, product aggregations)
3. **Temporal Awareness:** Strong hourly and daily seasonality patterns captured

#### Weaknesses:
1. **Store-level Performance:** Higher errors for low-volume stores
2. **Peak-hour Predictions:** More challenging to predict during peak trading hours
3. **Extrapolation:** May struggle with out-of-distribution values

### 💡 Key Insights:

1. **Temporal Patterns are Critical:**
   - Hourly patterns show strong cyclicality
   - Day-of-week effects are significant
   - Monthly seasonality impacts predictions

2. **Business Dimension Effects:**
   - Top stores have more consistent demand (easier to predict)
   - Small stores show higher volatility and prediction error
   - Product-level aggregations important for performance

3. **Error Distribution:**
   - Residuals show some systematic bias patterns
   - Errors cluster around specific business contexts (stores/times)
   - Opportunities for model refinement through segment-specific models

### 🎯 Recommendations for Improvement:

1. **Ensemble Approaches:**
   - Combine multiple model types to leverage complementary strengths
   - Time-based ensemble weighting (different models for different hours)
   - Store-specific model adaptation

2. **Feature Engineering:**
   - Add interaction terms (store × hour, product × day-of-week)
   - Implement lag features more comprehensively
   - Include external features (weather, events, promotions)

3. **Advanced Techniques:**
   - Gradient boosting models (XGBoost, LightGBM)
   - Hybrid statistical-ML approaches (SARIMA + ML)
   - Deep learning for complex temporal dependencies

4. **Operational Improvements:**
   - Segment forecasting by business dimensions
   - Implement online learning for model adaptation
   - Create specialized models for high-error segments